# Jacobians, Hessians & Matrix Calculus

## What's covered

- The **Jacobian** — gradient generalized to vector-valued functions
- The **Hessian** — second derivatives of a scalar function
- **Layout conventions** — denominator vs numerator layout (and which one this repo uses)
- **Matrix calculus identities** you'll use constantly — `∂(Ax)/∂x`, `∂(x^T A x)/∂x`, `∂||x||²/∂x`, and friends
- Worked example: **gradient of MSE** (recovered cleanly)
- Worked example: **gradient of softmax + cross-entropy** — the cleanest derivation in ML
- Where this appears in ML — backprop through linear layers, Hessian-based optimization, natural gradients


## From gradients to Jacobians

The **gradient** packaged partial derivatives of a *scalar-valued* function `f: R^n → R` into a vector. The **Jacobian** is the same idea for a *vector-valued* function.

If `\mathbf{f}: R^n \to R^m`, with components `f_1, f_2, \dots, f_m`, then its Jacobian is the `m × n` matrix:

$$
J_{\mathbf{f}}(\mathbf{x}) = \begin{bmatrix} \partial f_1 / \partial x_1 & \dots & \partial f_1 / \partial x_n \\ \vdots & & \vdots \\ \partial f_m / \partial x_1 & \dots & \partial f_m / \partial x_n \end{bmatrix}
$$

Each **row** is the gradient of one output component. Each **column** corresponds to one input variable. The shape — `(outputs) × (inputs)` — is the convention this repo follows. (See the next section on layout.)

**Three special cases.**

| `f` | `J_f` | shape | name |
|---|---|---|---|
| `R → R` | `f'(x)` | `1 × 1` | derivative |
| `R^n → R` | `(\nabla f)^T` | `1 × n` | gradient (transposed) |
| `R^n → R^m` | `J_f` | `m × n` | Jacobian |

The gradient and the Jacobian are *the same object*, viewed differently — the gradient is the row of the Jacobian when there's only one output. Conventions for "is the gradient a row or a column?" cause endless confusion; we use **column vectors** for gradients in prose and turn them into rows when stacking into a Jacobian.

**Geometric meaning.** The Jacobian is the *best linear approximation* of `f` near `x`. Locally, `\mathbf{f}(\mathbf{x} + \Delta \mathbf{x}) \approx \mathbf{f}(\mathbf{x}) + J_{\mathbf{f}} \, \Delta \mathbf{x}`. Notebook 6 (Taylor) makes this precise.


In [ ]:
import numpy as np

# A vector-valued function: f(x, y) = (x^2 + y, sin(x) * y)
def f(xy):
    x, y = xy
    return np.array([x ** 2 + y, np.sin(x) * y])

# Analytical Jacobian
def J_analytic(xy):
    x, y = xy
    return np.array([
        [2 * x,           1.0],
        [np.cos(x) * y,   np.sin(x)],
    ])

# Numerical Jacobian via finite differences
def J_numerical(f, xy, h=1e-5):
    xy = np.asarray(xy, dtype=float)
    n = xy.size
    J = np.zeros((f(xy).size, n))
    for j in range(n):
        ep, em = xy.copy(), xy.copy()
        ep[j] += h
        em[j] -= h
        J[:, j] = (f(ep) - f(em)) / (2 * h)
    return J

xy = np.array([1.0, 2.0])
print("J analytic:\n", J_analytic(xy))
print("J numerical:\n", J_numerical(f, xy))
print("max abs diff:", np.max(np.abs(J_analytic(xy) - J_numerical(f, xy))))


## Layout conventions — denominator vs numerator

A surprising amount of confusion in matrix calculus comes from one ambiguity: **is `∂y/∂x` shaped like the output, or like the input?** Two conventions coexist.

**Numerator layout** — `∂y/∂x` has the same shape as `y` (the numerator). The gradient of a scalar w.r.t. a vector is a *row* vector. Used in most physics / math textbooks.

**Denominator layout** — `∂y/∂x` has the same shape as `x` (the denominator). The gradient of a scalar w.r.t. a vector is a *column* vector. Used in most ML textbooks.

This repo uses **denominator layout**: gradients are columns, matching how PyTorch and JAX expose them. So if `L` is a scalar and `w` is a vector, `∂L/∂w` is a column vector with the same shape as `w`. For a Jacobian of a vector-valued function `\mathbf{f}: R^n \to R^m`, we keep it `m × n` (output × input) — this is the most common Jacobian convention and matches `numpy`/`scipy`/PyTorch's `jacobian` tools.

**Always sanity-check shapes.** Whatever convention you use, every matrix-calculus identity has a shape-only "test": if the shapes don't line up after applying the identity, the identity is wrong (or you mis-typed it). When in doubt, write the scalar-by-scalar definition first.


## The matrix calculus cheat sheet

These identities are the ones you'll use over and over. Each is straightforward to derive by writing out the scalar definition and re-stacking — but in practice you should memorize them. (All in **denominator layout**, gradients as columns, `A` constant unless stated.)

| Expression | Gradient w.r.t. `x` | Shape | Comment |
|---|---|---|---|
| `\mathbf{a}^T \mathbf{x}` | `\mathbf{a}` | `n × 1` | Linear form — gradient is the constant vector |
| `\mathbf{x}^T A \mathbf{x}` | `(A + A^T) \mathbf{x}` | `n × 1` | Quadratic form — symmetric `A` gives `2 A x` |
| `\|\mathbf{x}\|^2 = \mathbf{x}^T \mathbf{x}` | `2 \mathbf{x}` | `n × 1` | Special case of above with `A = I` |
| `\|\mathbf{x} - \mathbf{c}\|^2` | `2 (\mathbf{x} - \mathbf{c})` | `n × 1` | Squared distance to a point |
| `\mathbf{x}^T \mathbf{a}` (scalar) | `\mathbf{a}` | `n × 1` | Same as first row, written the other way |

For Jacobians of vector-valued operations:

| Expression | Jacobian w.r.t. `x` | Shape |
|---|---|---|
| `A \mathbf{x}` | `A` | `m × n` |
| `\mathbf{x}` (identity) | `I` | `n × n` |
| `f(\mathbf{x})` element-wise | `\text{diag}(f'(x_1), \dots, f'(x_n))` | `n × n` |

For products and inverses (`A` depends on `x`):

| Expression | Derivative | Comment |
|---|---|---|
| `\text{tr}(A^T B)` | `B` (w.r.t. `A`) | Trace identities are the secret to matrix gradients |
| `\log \det A` | `(A^{-1})^T` (w.r.t. `A`) | Shows up in Gaussian likelihoods and normalizing flows |
| `A^{-1}` (w.r.t. `A`) | `-A^{-1} (\cdot) A^{-1}` | Used in implicit-function-theorem derivations |

**Most useful in practice:** rows 1 and 2 of the first table. Linear regression, ridge regression, every linear-layer backward pass — all rest on these two identities.


In [ ]:
# Verify the two most-used identities

n = 4
rng = np.random.default_rng(0)
A = rng.normal(size=(n, n))
a = rng.normal(size=n)
x = rng.normal(size=n)

# 1) ∂(a^T x) / ∂x = a
def f1(x): return a @ x
g1_analytic = a
g1_numeric  = np.array([(f1(x + 1e-5 * np.eye(n)[i]) - f1(x - 1e-5 * np.eye(n)[i])) / 2e-5 for i in range(n)])
print("(1) grad(a^T x)       analytic:", g1_analytic.round(6))
print("                       numeric:", g1_numeric.round(6))

# 2) ∂(x^T A x) / ∂x = (A + A^T) x
def f2(x): return x @ A @ x
g2_analytic = (A + A.T) @ x
g2_numeric  = np.array([(f2(x + 1e-5 * np.eye(n)[i]) - f2(x - 1e-5 * np.eye(n)[i])) / 2e-5 for i in range(n)])
print("\n(2) grad(x^T A x)     analytic:", g2_analytic.round(6))
print("                       numeric:", g2_numeric.round(6))

# Special case: symmetric A ⇒ gradient is 2 A x
A_sym = (A + A.T) / 2
def f3(x): return x @ A_sym @ x
print("\n(3) grad(x^T A_sym x) analytic (2 A x):", (2 * A_sym @ x).round(6))
print("                       numeric        :", np.array([(f3(x + 1e-5 * np.eye(n)[i]) - f3(x - 1e-5 * np.eye(n)[i])) / 2e-5 for i in range(n)]).round(6))


## The Hessian

The **Hessian** of a scalar function `f: R^n → R` is the matrix of all second-order partial derivatives:

$$
H_f(\mathbf{x})_{ij} = \frac{\partial^2 f}{\partial x_i \, \partial x_j}
$$

`H_f` is `n × n`. **Clairaut's theorem** (mixed partials commute, when continuous) tells us `H_f` is **symmetric** — same content above and below the diagonal.

**Two ways to think of `H_f`.**

- As the **Jacobian of the gradient**: take `∇f: R^n → R^n` and Jacobian it.
- As the **quadratic-coefficient matrix** in the Taylor expansion: locally, `f(x + \Delta) \approx f(x) + \nabla f(x)^T \Delta + \tfrac{1}{2} \Delta^T H_f(x) \, \Delta`. We do this carefully in notebook 6.

**Geometric meaning at a critical point** (where `∇f = 0`):

- `H_f` positive definite (all eigenvalues > 0) → local **minimum** (bowl).
- `H_f` negative definite (all eigenvalues < 0) → local **maximum** (dome).
- `H_f` indefinite (mixed signs) → **saddle point** (a pringle).
- `H_f` semidefinite or singular → degenerate, need higher-order analysis.

These are the **second-order conditions** for optimization (notebook 7). They are the lens through which Newton's method, BFGS, L-BFGS, and natural gradient all justify themselves.

**Cost.** For a model with `n` parameters, `H_f` is `n × n` — storing it requires `O(n²)` memory and inverting it requires `O(n³)` time. For a modern network with billions of parameters this is **impossible**. ML usually settles for Hessian *approximations* (diagonal, block-diagonal, low-rank, or Hessian-vector products).


In [ ]:
# Hessian of f(x, y) = x^2 + 2 y^2 + x y (our friend from notebook 3)
def f(xy):
    x, y = xy
    return x ** 2 + 2 * y ** 2 + x * y

# Analytical Hessian — constant for a quadratic
H_analytic = np.array([[2.0, 1.0],
                       [1.0, 4.0]])

# Numerical Hessian via central differences
def numerical_hessian(f, x, h=1e-4):
    n = x.size
    H = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            ei = np.eye(n)[i]
            ej = np.eye(n)[j]
            H[i, j] = (f(x + h*ei + h*ej) - f(x + h*ei - h*ej)
                       - f(x - h*ei + h*ej) + f(x - h*ei - h*ej)) / (4 * h * h)
    return H

H_num = numerical_hessian(f, np.array([1.0, 1.0]))
print("Hessian analytic:\n", H_analytic)
print("\nHessian numerical:\n", H_num.round(6))

# Eigenvalues tell us this is a minimum (both positive)
eigvals = np.linalg.eigvalsh(H_analytic)
print(f"\neigenvalues of H: {eigvals}  → both > 0, so (0, 0) is a local minimum")


## Worked example — gradient of MSE (revisited cleanly)

Notebook 3 derived the MSE gradient with effort. Now with the identity table we get it in *three lines*.

$$
L(\mathbf{w}) = \frac{1}{n} \|\mathbf{y} - X \mathbf{w}\|^2 = \frac{1}{n} (\mathbf{y} - X\mathbf{w})^T (\mathbf{y} - X\mathbf{w})
$$

Expand:

$$
L = \frac{1}{n} \left( \mathbf{y}^T \mathbf{y} - 2 \mathbf{w}^T X^T \mathbf{y} + \mathbf{w}^T X^T X \mathbf{w} \right)
$$

Differentiate term by term using the identities. `\mathbf{y}^T \mathbf{y}` doesn't depend on `w`, gradient zero. `\mathbf{w}^T X^T \mathbf{y} = (X^T \mathbf{y})^T \mathbf{w}` — linear form, gradient is `X^T \mathbf{y}`. `\mathbf{w}^T (X^T X) \mathbf{w}` — quadratic form with **symmetric** `X^T X`, gradient is `2 X^T X \mathbf{w}`. Combine:

$$
\nabla_{\mathbf{w}} L = \frac{2}{n} X^T (X \mathbf{w} - \mathbf{y})
$$

The Hessian is just as clean — it's `2/n \cdot X^T X`, *constant*, symmetric, PSD. That's why linear regression is convex and why the closed-form solution exists.


## Worked example — softmax + cross-entropy

This is the single cleanest gradient in ML, and the one most often asked in interviews.

**Softmax** for a logit vector `\mathbf{z} \in R^K`:

$$
p_i = \frac{e^{z_i}}{\sum_k e^{z_k}}
$$

**Cross-entropy loss** against a one-hot target `\mathbf{y}` (entry `y_t = 1` for the true class `t`, zero elsewhere):

$$
L = -\sum_i y_i \log p_i = -\log p_t
$$

**Goal.** Compute `∂L/∂z_i`. Two derivatives are needed: `∂p_i / ∂z_j` (softmax's own Jacobian) and the chain rule with the log.

The softmax Jacobian, derived in the cell below:

$$
\frac{\partial p_i}{\partial z_j} = p_i (\delta_{ij} - p_j)
$$

(where `δ_{ij}` is 1 if `i = j` else 0). Now the cross-entropy gradient (chain rule):

$$
\frac{\partial L}{\partial z_j} = -\sum_i y_i \frac{1}{p_i} \cdot p_i (\delta_{ij} - p_j) = -\sum_i y_i (\delta_{ij} - p_j)
$$

Distribute and use `Σ y_i = 1`:

$$
\frac{\partial L}{\partial z_j} = -y_j + p_j \sum_i y_i = p_j - y_j
$$

So the gradient of softmax cross-entropy with respect to the logits is

$$
\boxed{\nabla_{\mathbf{z}} L = \mathbf{p} - \mathbf{y}}
$$

**Prediction minus target.** That's the entire backward pass through softmax + cross-entropy. This identity is why classification networks are so easy to train: the gradient is meaningful, bounded, and trivially computable.


In [ ]:
# Verify ∇_z L = p - y for softmax + cross-entropy
def softmax(z):
    z = z - z.max()    # numerical stability (the log-sum-exp trick from notebook 1)
    e = np.exp(z)
    return e / e.sum()

def cross_entropy(z, y_onehot):
    p = softmax(z)
    return -np.sum(y_onehot * np.log(p + 1e-12))

K = 5
rng = np.random.default_rng(0)
z = rng.normal(size=K)
t = 2                       # true class
y_onehot = np.zeros(K); y_onehot[t] = 1

# Analytical gradient
p = softmax(z)
g_analytic = p - y_onehot

# Numerical gradient
g_num = np.zeros(K)
for i in range(K):
    zp, zm = z.copy(), z.copy()
    zp[i] += 1e-5; zm[i] -= 1e-5
    g_num[i] = (cross_entropy(zp, y_onehot) - cross_entropy(zm, y_onehot)) / 2e-5

print("analytic p - y:", g_analytic.round(6))
print("numerical    :", g_num.round(6))
print("max abs diff :", np.max(np.abs(g_analytic - g_num)))


## Where this appears in ML

Matrix calculus is the language every deep-learning paper is written in, and the cheat sheet above shows up in every backward pass.

- **Backprop through a linear layer.** `y = W x + b`. Then `∂L/∂W = (∂L/∂y) x^T`, `∂L/∂x = W^T (∂L/∂y)`, `∂L/∂b = ∂L/∂y`. Each is a one-line application of an identity above.
- **Softmax + cross-entropy.** `∇_z L = p - y` is one of the most-used facts in ML — it's *why* classification networks train cleanly.
- **MSE gradient.** Direct application of `∂(x^T A x)/∂x = 2 A x` and `∂(a^T x)/∂x = a`.
- **Ridge / weight decay.** Adding `λ ||w||^2` to a loss adds `2 λ w` to the gradient. That's why weight decay is "just" rescaling weights toward zero.
- **Hessian-based optimization.** Newton's method (`w ← w - H^{-1} \nabla L`), L-BFGS (low-memory Hessian approximation), natural gradient (`H` replaced by the Fisher information matrix). All notebook 6 and 7 material.
- **Gauss-Newton and Levenberg-Marquardt.** Use the Jacobian of the residual function to approximate the Hessian for least-squares problems.
- **Trust-region methods.** Bound how far you trust the local quadratic approximation `f + g^T s + (1/2) s^T H s`.
- **Influence functions.** Compute how a single training point would shift the model's parameters — formula involves `H^{-1}`.
- **Normalizing flows.** Train likelihood `log p(x) = log p(z) + log |det J_f(x)|` — the `log det Jacobian` is the heart of every flow architecture.
- **Implicit function theorem (in MAML, OptNet, DEQ).** Differentiate through fixed-point or argmin operations using `-H^{-1} \nabla`.

Next notebook: **Taylor series and approximations** — we make the "locally `f` looks like a line" intuition precise, derive Newton's method properly, and preview Laplace approximation.
